In [2]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [3]:
from config import PER_ORGANISM_PER_CLASS_ALLELES
from mhcgnomes import parse

DATA_PATH = "/home/denklewer/Data/schreiber_full/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-IAd', 'H-2-Ld', 'H-2-IEd', 'H-2-Dd', 'H-2-Kd']

In [4]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [5]:
METHOD = 'mhcflurry'

## 0.1. make result dicts

In [6]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [7]:
per_allele_decoy = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    
    per_allele_decoy[allele_name] = decoy_df.copy()

# 1. Predict with mhcflurry

In [8]:
# importing the predict module
from mhcflurry import Class1PresentationPredictor
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

In [9]:
from mhcgnomes import parse

# different allele naming.
TARGET_ALLELES = dict()

## allele normalization now with mhcgnomes
for allele_name in ALLELES:
    result = parse(allele_name)
    """
    parse(
        raw_name,
        only_class1=True,
        required_result_types=[Allele, AlleleWithoutGene, Gene],
        preferred_result_types=[Allele],
        use_allele_aliases=True,
        infer_class2_pairing=False,
        collapse_singleton_haplotypes=True,
        collapse_singleton_serotypes=True,
        raise_on_error=False)
    """
    target_allele_name = result.to_string()
    TARGET_ALLELES[allele_name] = target_allele_name


TARGET_ALLELES

{'H-2-IAd': 'H2-AA*d-AB*d',
 'H-2-Ld': 'H2-L*d',
 'H-2-IEd': 'H2-EA*d-EB*d',
 'H-2-Dd': 'H2-D*d',
 'H-2-Kd': 'H2-K*d'}

In [10]:
def filterByLength(per_allele_df, length_column='length'):
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    for allele_name in per_allele_df.keys():
        
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 15]
        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
        

    
print(f"\t mt")
filterByLength(per_allele_mt_df)
print(f"\t wt")
filterByLength(per_allele_wt_df)

	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples


In [15]:
def predict_with_mhcflurry(per_allele_df , per_allele_ref_scores=None):
    per_allele_predictions = dict()
    
    predictor = Class1PresentationPredictor.load()
    supported_alleles =  predictor.supported_alleles
    
    for allele_name in ALLELES:
        df =  per_allele_df[allele_name].copy()
        target_allele = TARGET_ALLELES[allele_name]
        print(f"---{target_allele}")
        if target_allele in supported_alleles:

            peptides = df[df['peptide'].notnull()]['peptide'].values
            # affinity	best_allele	affinity_percentile	processing_score	presentation_score	presentation_percentile
            prediction_df = predictor.predict(alleles=[target_allele], peptides=peptides, include_affinity_percentile=True)
            result_df = df.copy()
            result_df.loc[result_df['peptide'].notnull(), 'affinity'] = prediction_df['affinity']
            result_df.loc[result_df['peptide'].notnull(), 'percentile_rank_ba'] = prediction_df['affinity_percentile']
            result_df.loc[result_df['peptide'].notnull(), 'processing_score'] = prediction_df['processing_score']
            result_df.loc[result_df['peptide'].notnull(), 'score_el'] = prediction_df['presentation_score']
            result_df.loc[result_df['peptide'].notnull(), 'percentile_rank_el'] = prediction_df['presentation_percentile']
            
            
            # calculate percentile ranks if mhcflurry doesn't have such scores for allele
            allele_to_percent_rank_transform = predictor.affinity_predictor.allele_to_percent_rank_transform
            if target_allele not in allele_to_percent_rank_transform:
                if per_allele_ref_scores:
                    print("Calculate percentile ranks")
                    scores  = 1 - np.log(result_df['affinity'])/np.log(50000)
                    ref_scores = 1 - np.log(per_allele_ref_scores[allele_name]['affinity'])/np.log(50000)
                    ref_scores = ref_scores.sort_values()
                    pct_increment = 1./len(ref_scores)
                    percentile_ranks = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
                    percentile_ranks = 100 - percentile_ranks*100
                    result_df['percentile_rank_ba'] = percentile_ranks                
        else:
            #dummy prediction
            print("Method doesn't support {}".format(target_allele))
            result_df = df.copy()
            result_df['affinity'] = np.nan
            result_df['percentile_rank_ba'] = np.nan
            result_df['processing_score'] = np.nan
            result_df['score_el'] = np.nan
            result_df['percentile_rank_el'] = np.nan
        per_allele_predictions[allele_name] = result_df
        
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



## 0.2 Read decoy data

In [17]:
sample_predictor = Class1PresentationPredictor.load()

per_allele_decoy = dict()
for allele_name in ALLELES:
    target_allele = TARGET_ALLELES[allele_name]
    allele_to_percent_rank_transform = sample_predictor.affinity_predictor.allele_to_percent_rank_transform
    if target_allele not in allele_to_percent_rank_transform:
        organism =  defineOrganism(allele_name).lower()
        class_to_use = defineClass(allele_name)
        DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
        decoy_df =  pd.read_csv(DECOY_DATA_PATH)
        decoy_df['length'] = decoy_df['peptide'].str.len()

    per_allele_decoy[allele_name] = decoy_df.copy()

In [20]:
%%time
print("decoy ")
filterByLength(per_allele_decoy)


decoy 
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
CPU times: user 72.1 ms, sys: 4.24 ms, total: 76.3 ms
Wall time: 73.1 ms


## First calculate scores for decoy

In [21]:
%%time
per_allele_decoy_predictions = predict_with_mhcflurry(per_allele_decoy)

  0%|          | 0/2 [00:00<?, ?it/s]

---H2-AA*d-AB*d
Method doesn't support H2-AA*d-AB*d
---H2-L*d
Predicting processing.


Instructions for updating:
The `validate_indices` argument has no effect. Indices are always validated on CPU and never validated on GPU.
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Mode

Predicting affinities.


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
100%|██████████| 1/1 [01:24<00:00, 84.62s/it]
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-L*d has no percentile rank information
  warnings.warn(msg)
  0%|          | 0/2 [00:00<?, ?it/s]

---H2-EA*d-EB*d
Method doesn't support H2-EA*d-EB*d
---H2-D*d
Predicting processing.


... (5 outputs left out)

Predicting affinities.


100%|██████████| 1/1 [01:15<00:00, 75.77s/it]


CPU times: user 16min 14s, sys: 54 s, total: 17min 8s
Wall time: 9min 12s


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-K*d has no percentile rank information
  warnings.warn(msg)


In [26]:
per_allele_decoy_predictions[ALLELES[4]]

,peptide,length,affinity,percentile_rank_ba,processing_score,score_el,percentile_rank_el
0,NRTYSFQR,8,31438.907770,NaN,0.240352,0.007504,31.643179
1,AFYSGHGA,8,17211.455841,NaN,0.399518,0.023283,10.231984
2,AKKQSLLD,8,32697.730681,NaN,0.028266,0.003425,99.286603
3,RWLVVNVAHDLL,12,24818.531625,NaN,0.196215,0.008072,27.617391
4,ARRGDLDVLRSL,12,22667.826575,NaN,0.119942,0.006739,37.359049
...,...,...,...,...,...,...,...
199995,PVINDILTLALS,12,31110.669026,NaN,0.012550,0.003400,99.286603
199996,QIFTGNVPGAG,11,26326.775713,NaN,0.030820,0.004262,62.744674
199997,PRESHDKN,8,31829.659909,NaN,0.000332,0.003185,99.286603
199998,IHSGAKPYSC,10,19724.129590,NaN,0.026339,0.005546,46.224674


In [27]:
%%time
print(f"--mt")
per_allele_mt_predictions = predict_with_mhcflurry(per_allele_mt_df, per_allele_decoy_predictions)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  

print(f"--wt")
per_allele_wt_predictions = predict_with_mhcflurry(per_allele_wt_df, per_allele_decoy_predictions)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt


  0%|          | 0/1 [00:00<?, ?it/s]

---H2-AA*d-AB*d
Method doesn't support H2-AA*d-AB*d
---H2-L*d
Predicting processing.


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automat

Predicting affinities.


100%|██████████| 1/1 [00:31<00:00, 31.03s/it]
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-L*d has no percentile rank information
  warnings.warn(msg)
  0%|          | 0/1 [00:00<?, ?it/s]

... (20 outputs left out)

Predicting affinities.


100%|██████████| 1/1 [00:03<00:00,  3.50s/it]

Calculate percentile ranks
CPU times: user 4min 17s, sys: 7.81 s, total: 4min 24s
Wall time: 3min 36s



/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-K*d has no percentile rank information
  warnings.warn(msg)


In [30]:
per_method_per_allele_wt_predictions[METHOD][ALLELES[3]]

,source_sequence_name,allele,peptide,length,affinity,percentile_rank_ba,processing_score,score_el,percentile_rank_el
0,seq124,H-2-Dd,SALSILLL,8,13139.408338,6.30275,0.168184,0.013483,17.305598
1,seq194,H-2-Dd,SVTAVAETQ,9,24284.059134,36.56725,0.024910,0.004513,62.744674
2,seq329,H-2-Dd,SVTAVAET,8,29054.427592,80.58125,0.020173,0.003731,99.286603
3,seq401,H-2-Dd,GGDCLLLL,8,16777.426978,11.19775,0.022058,0.006387,37.359049
4,seq431,H-2-Dd,DVINSRTQ,8,28488.109146,74.20725,0.173762,0.006531,37.359049
...,...,...,...,...,...,...,...,...,...
8733,seq26209,H-2-Dd,MGVSSLQGIQF,11,7001.668319,1.91825,0.016898,0.014535,16.157554
8734,seq26210,H-2-Dd,GVSSLQGIQFF,11,24288.535463,36.59825,0.053523,0.004991,62.744674
8735,seq26211,H-2-Dd,VSSLQGIQFFD,11,28611.630001,75.57375,0.001378,0.003544,99.286603
8736,seq26212,H-2-Dd,VSSLQGIQFF,10,12232.351315,5.41425,0.023988,0.008720,27.617391


# 2. Join predictions

In [31]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank'], 
    'mhcflurry':['source_sequence_name', 'affinity', 'percentile_rank_ba', 'processing_score', 'score_el', 'percentile_rank_el']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [32]:
def joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print(f"method {method}")
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                
                

              

In [33]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2-IAd
method mhcflurry
allele H-2-Ld
method mhcflurry
allele H-2-IEd
method mhcflurry
allele H-2-Dd
method mhcflurry
allele H-2-Kd
method mhcflurry


# 5. For now just write to files

In [34]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [35]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [36]:
total_df

,source_sequence_name,allele,peptide_mt,length,mhcflurry_affinity_mt,mhcflurry_percentile_rank_ba_mt,mhcflurry_processing_score_mt,mhcflurry_score_el_mt,mhcflurry_percentile_rank_el_mt,peptide_wt,mhcflurry_affinity_wt,mhcflurry_percentile_rank_ba_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcflurry_percentile_rank_el_wt
0,seq0,H-2-Kd,KYLTQRENL,9,51.458174,0.02275,0.766347,0.961114,0.031549,KYLTQREKL,55.655188,0.02975,0.732204,0.953047,0.042717
1,seq6,H-2-Kd,VYQVERERL,9,87.694444,0.06975,0.749491,0.932768,0.073397,VYQVELERL,70.240004,0.04575,0.826407,0.957618,0.036957
2,seq11,H-2-Kd,RFAPQAELL,9,123.872843,0.10375,0.638161,0.869961,0.168750,RFPPQAELL,121.563169,0.10175,0.401535,0.746812,0.362446
3,seq14,H-2-Kd,APSNSGAGL,9,10626.824493,5.32275,0.237200,0.020992,11.630326,APSNSGADL,13596.063892,7.43725,0.155574,0.012487,18.617989
4,seq15,H-2-Kd,SFVAVVITI,9,103.092692,0.08575,0.273558,0.687609,0.459891,SFVAAVITI,89.399280,0.07275,0.470065,0.835162,0.222663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,seq14195,H-2-Kd,PTTVCQPN,8,31681.236927,93.94425,0.000591,0.003203,99.286603,ATTVCQPN,31587.679473,93.24625,0.001968,0.003227,99.286603
8734,seq14197,H-2-Kd,PRLHHSSR,8,30014.279997,77.64425,0.002694,0.003400,99.286603,PLLHHSSR,29461.855873,71.46875,0.005839,0.003500,99.286603
8735,seq14200,H-2-Kd,PPRLHHSSRSC,11,29377.141929,70.53175,0.001324,0.003454,99.286603,PPLLHHSSRSC,28689.863950,63.03625,0.001152,0.003532,99.286603
8736,seq14201,H-2-Kd,PNNISAIS,8,31589.145213,93.25975,0.002954,0.003238,99.286603,PNNISGIS,31438.486048,92.08225,0.001649,0.003239,99.286603
